# LD-1 — LINE-DETR : entraînement de contrôle Z-GT sans CTC (GEL 2, GPU)

In [ ]:

import time
T_START = time.monotonic()          # §16 (3) : pris au tout début de la cellule
import datetime, faulthandler, hashlib, json, os, shutil, subprocess, sys
from pathlib import Path
faulthandler.enable()

def _utc():
    return datetime.datetime.now(datetime.timezone.utc).strftime('%H:%M:%S')

print('ld1 start', _utc(), flush=True)
OUT = Path("/kaggle/working")
KIT_CANDIDATES = ['/kaggle/input/babaorum-ld1-kit', '/kaggle/input/datasets/mabaku/babaorum-ld1-kit']
PAGES_CANDIDATES = ['/kaggle/input/babaorum-e2e3-run3-fp-classes/e2e3/mf', '/kaggle/input/notebooks/mabaku/babaorum-e2e3-run3-fp-classes/e2e3/mf']
CKPT_CANDIDATES = ['/kaggle/input/babaorum-e2e5-lambda/logs/checkpoint_last.pt', '/kaggle/input/notebooks/mabaku/babaorum-e2e5-lambda/logs/checkpoint_last.pt']
INIT_SHA = 'ef5e826803c347f73e49c83c0f21f61399f9ea06ee985905bd17644dff3844fa'
TIMEOUTS = {'D': 900, 'T': 600, 'E': 4500, 'V': 900, 'pytest_heavy': 5400}
LIMITS = {'deadline_s': 7200, 'v_reserve_s': 900, 'stop_margin_s': 120, 'e_reserve_s': 300}
def resolve_one(candidates, marker):
    """Exactement un candidat dont <candidat>/<marqueur> existe. Aucun listage."""
    hits = [Path(c) for c in candidates if (Path(c) / marker).is_file()]
    if len(hits) != 1:
        raise RuntimeError(f"montage : {len(hits)} candidat(s) pour {marker}")
    return hits[0]

def resolve_file(candidates):
    """Exactement un candidat qui est un fichier (§12 (15))."""
    hits = [Path(c) for c in candidates if Path(c).is_file()]
    if len(hits) != 1:
        raise RuntimeError(f"fichier : {len(hits)} candidat(s)")
    return hits[0]


def sha256(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

KIT = resolve_one(KIT_CANDIDATES, "config/ld1_line_detr.json")
PAGES = resolve_one(PAGES_CANDIDATES, "pages/mf_0000.json")
CKPT = resolve_file(CKPT_CANDIDATES)
assert INIT_SHA != "NON FIXÉ", "sha complet du checkpoint non fixé"
assert sha256(CKPT) == INIT_SHA, "checkpoint : sha différent"
sys.path.insert(0, str(KIT / "src"))
sys.path.insert(0, str(KIT / "scripts"))
import importlib.util as _u
_spec = _u.spec_from_file_location("data2_common", KIT / "scripts" / "data2_common.py")
C = _u.module_from_spec(_spec); _spec.loader.exec_module(C)

def _argv(phase, extra=()):
    return [sys.executable, str(KIT / "scripts" / "train_line_detr.py"), "--phase", phase,
            "--root", str(PAGES), "--init-checkpoint", str(CKPT), "--device", "cuda",
            "--cache-dir", "/tmp/ld1_hf", *extra]

RUN = OUT / "ld1"
EXITS = OUT / "exits"
class Incomplete(Exception):
    pass

def remaining():
    return LIMITS["deadline_s"] - (time.monotonic() - T_START)

def allowed(cap, reserve=0.0):
    """§16 (3) : max(0, min(plafond, restant - réserve))."""
    return max(0.0, min(cap, remaining() - reserve))

def step_argv(ph, build_argv, cap, reserve=0.0, **kw):
    """Étape argv dont le délai est calculé au lancement ; délai nul : INCOMPLETE sans lancer l'étape."""
    def argv():
        t = allowed(cap, reserve)
        if t <= 0:
            raise Incomplete(f"échéance globale atteinte avant l'étape {ph}")
        st["timeout_s"] = t
        return build_argv()
    st = {"name": ph, "argv": argv, "timeout_s": cap, **kw}
    return st

def s_budget():
    """§15 (4) : S recalculé sur le budget réellement disponible pour E, écrit au manifeste AVANT E."""
    mf = RUN / "ld1_timing_manifest.json"
    man = json.loads(mf.read_text())
    budget_e = allowed(TIMEOUTS["E"], LIMITS["v_reserve_s"] + LIMITS["stop_margin_s"])
    S = min(man["max_steps"], int((budget_e - LIMITS["e_reserve_s"]) // man["s_per_step"]))
    if budget_e <= 0 or S < 1:
        raise Incomplete("échéance globale : budget insuffisant pour l'étape E")
    man.update(steps=S, budget_s=budget_e, e_reserve_s=LIMITS["e_reserve_s"])
    mf.write_text(json.dumps(man, indent=1))

def ld1_cleanup(rec):
    """TIMEOUT, échéance ou échec d'étape : INCOMPLETE à l'étape, sauf si l'étape a écrit FAILED (device, garde)."""
    if rec["status"] == "TIMEOUT" or str(rec.get("exception") or "").startswith("Incomplete:"):
        rec["status"] = "INCOMPLETE"
    elif rec["status"] == "FAILED" and rec["exit_code"] != "EXCEPTION":
        f = EXITS / f"ld1_run_exit_{rec['step']}.json"
        if not f.is_file() or json.loads(f.read_text()).get("status") != "FAILED":
            rec["status"] = "INCOMPLETE"

def s_controls():
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA indisponible : run complet refusé")
    from importlib.metadata import version as _v
    (OUT / "ld1_env.json").write_text(json.dumps({     # échec (PackageNotFoundError) si transformers ou huggingface_hub absents
        "python": sys.version, "torch": torch.__version__, "cuda": torch.version.cuda, "numpy": _v("numpy"),
        "PIL": _v("pillow"), "transformers": _v("transformers"), "huggingface_hub": _v("huggingface_hub")}, indent=1))
    from babaorum.experimental.line_detr import LineDetr, load_enc_det_from_e2e5
    ck = torch.load(CKPT, map_location="cpu")
    load_enc_det_from_e2e5(LineDetr(), ck["model"], expected_step=2000, step=ck["step"])    # init_check
    print("controls ok", _utc(), flush=True)

STEPS = [{"name": "controls", "fn": s_controls}]
OFFLINE = lambda: dict(os.environ, HF_HUB_OFFLINE="1")       # §13 (3) : hors ligne sauf pendant D
RESERVES = {"D": 0.0, "T": 0.0, "E": LIMITS["v_reserve_s"] + LIMITS["stop_margin_s"], "V": 0.0}
STEPS += [step_argv(ph, (lambda ph=ph: _argv(ph, ["--out", str(RUN)])), TIMEOUTS[ph], RESERVES[ph],
                    **({} if ph == "D" else {"env": OFFLINE})) for ph in ("D", "T", "E", "V")]
STEPS.insert(3, {"name": "s_budget", "fn": s_budget})        # controls, D, T, s_budget, E, V
EXIT = None
try:
    EXIT = C.execute_pipeline(OUT, STEPS, heartbeat_interval_s=60.0, cleanup=ld1_cleanup)
finally:
    (OUT / "ld1_run_exit.json").write_text(json.dumps(EXIT if EXIT is not None else {"status": "FAILED_BEFORE_EXIT"},
                                                      indent=1, default=str))
print("EXIT", json.dumps({k: EXIT[k] for k in ("status", "step", "exit_code")}) if EXIT else "none", flush=True)
